# TaskManager 探针后的源事件回归门槛 v1

依据已冻结的 v2 自动失败产物及单轮聚焦修复探针进行离线审计。本工作未调用模型，未修改冻结的 `.san` 文件，属于复用开发样本的规则设计，不属于独立正式测试。详情见 `data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_event_guard_v1_review.json`。


## 区分改名与无法追踪的绑定

SSM 中 24 条可调用源事件表达式在 v2 输入产物中均能精确匹配；另有 3 条源赋值表达式不纳入本次静态同名匹配。探针候选中 1 条精确匹配、22 条可解释为 `handle` 前缀改名（如 `onStatClick(stat)` → `handleStatClick(stat)`，参数仍在）、1 条重复的 `click:closeModal` 无法按此规则匹配。故不能笼统地称 22 个功能丢失，也不能凭相似命名宣称功能等价。探针中存在的无处理器修饰符及内联事件写法仍是独立的语法问题。


## 实现、边界与验证

新增 `migration_pipeline/utils/repair_regression_guard.py`，仅在聚焦语义修复候选完成校验后运行：新增无法追踪的源绑定判为 reject，可追踪的改名判为 review；两者均不自动计为通过，并阻止继续消耗修复轮次。被拒候选仍保留为失败证据，停止原因为 `repair_event_contract_regression`；非聚焦修复路径不受影响。聚焦 Prompt 同时要求保留有效源处理器名及参数；Enter 等按键修饰符可转为 San 的 `enter:` 值修饰符或等价具名守卫方法，内联子组件事件转发必须保留 payload。

确定性语法转换分两类：简单的 `on-event.stop/prevent="method(item.id)"`、`on-click.stop="fire('edit', task)"` 和键盘 `on-keyup.enter="addComment"` 转为 San 值修饰符，并保留调用与参数；真正没有处理器的 `on-event.stop/prevent` 转为带组件作用域空方法的值修饰符。`$event`、`dispatch/emit`、非键盘 `.enter`、已有值的复杂表达式不自动改写。回归门槛能识别 San 值修饰符，不把 `on-click="stop:choose(item.id)"` 误判为源绑定丢失。

对冻结探针候选只读重放：空修饰符规则修正 4 处，在父组件和内联子组件各添加 1 个空方法；新增简单值修饰符规则处理两处 `TaskItem` 的 `stop:fire(...)` 和一处 `TaskDetail` 的 `enter:addComment`。静态非法事件绑定由 7 处降至 0，San 编译通过；但 `missing_event_handlers`、`event_arguments_not_landed`、`lifecycle_behavior_not_landed`、`missing_event_bindings` 仍在。这仅是语法修正，未证明运行时等价或当前组件通过；此前冻结的开发集人工通过产物虽使用同类值修饰符，也不能替代本次验证。全量单测 180/180 通过。后续应单独核查源事件改名和赋值表达式、子组件 payload 转发、生命周期时序，再考虑冻结并运行新一轮单样本探针。


## 单轮语义探针 v2 的运行前冻结

仍从已冻结的 TaskManager v2 开发失败产物出发，先在内存中应用有限的事件语法归一化，保留原始源处理器和参数；预处理后的静态错误仍包含 `invalid_san_event_handler_syntax`（源赋值事件）、`invalid_san_computed_instance_method_call`、`hidden_template_reactive_dependency`、`nested_state_mutation` 和 `lifecycle_behavior_not_landed`。这五类错误须交给一次聚焦语义修复，不将语法预处理视为迁移通过。

已通过 `migration_pipeline/probe_formal_30_v4_task_manager_semantic_focus_v2.py --freeze` 冻结输入、Prompt、错误列表和实现文件，目录为 `data/experiments/snapshots/formal_30_v4_task_manager_semantic_probe_v2_pre_run_20261028`，18 个 payload 哈希校验通过。输入 SHA-256 为 `ff7d78aae16174bf1d08e694a5c2e122686fc5544a93725fa6d68b94f9dc063f`，Prompt SHA-256 为 `7e1ee5477fcc866580a1179ff2c043dcb1b582bd4a11a769d0a6fc00737a8ee9`。探针使用真实 RepairStage 后处理和源事件回归门槛，禁止覆盖已有输出；干跑通过，全量单测 181/181 通过。这是 v2 初始运行前记录，后续预检失败与改版另见下一节。


## v2 预检失败及 v2b 修正

用户执行 v2 `--execute` 后，程序在模型请求前抛出 `Current input/prompt differs from pre-run freeze`。诊断显示冻结文件与内存输入/Prompt 的文本内容完全相同、源快照相同、输出和报告均不存在；差异来自 Windows 文本写入将 LF 变为 CRLF：冻结记录使用文件字节哈希，执行前却误与内存规范化文本哈希比较。这是探针预检实现错误，不是模型失败，也没有产生模型调用或候选结果。v2 运行前快照保留，不修改。

修正为：冻结文件原始字节哈希与该文件重新计算的字节哈希比较；另将读回的文本与当前内存文本比较；继续校验实现文件哈希。新增预检回归测试，完整单测 182/182 通过。使用独立 v2b 目录重新冻结 18 个 payload：`data/experiments/snapshots/formal_30_v4_task_manager_semantic_probe_v2b_pre_run_20261028`。只读预检通过、v2b 输出和报告路径均为空；输入及 Prompt 的规范化文本 SHA-256 未变。此时仍未调用模型，未作通过结论。


## v2b 单轮探针结果：语法改善但事件回归拒绝

用户完成 v2b 一次修复请求，使用 qwen-plus，总计 23,568 tokens。结果先冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_semantic_probe_v2b_result_20261028_181549`，10 个 payload 哈希校验通过。静态验证未通过，视觉评估通过且结构相似度 0.9889；事件回归门槛为 reject，故自动通过为 false，未进行浏览器人工验证。这是复用开发样本的诊断结果，不是独立测试集成绩。机器可读审计见 `data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_semantic_probe_v2b_review.json`。

与 v1 探针相比，`invalid_san_event_handler_syntax` 已消除，但 `nested_state_mutation` 仍存在：`moveTask` 对 `this.data.get('tasks')` 返回数组直接 `splice`，未回写新数组。源 Vue 的 `mounted()` 依次调用 `loadInitialData()`、`startAutoSync()`、`showNotification()`；候选把第一项放入 `inited()`、后两项放入 `attached()`，故生命周期时序仍未按要求落地。

候选再次将 22 条源处理器改为 `handle` 前缀：多数对应参数仍可在候选绑定中看到，因此不能把所有名称不匹配解释为真实参数丢失；也不能凭命名相似认定运行等价。另有一条确实无法追踪：通知关闭按钮绑定 `handleHideNotification`，脚本只定义 `hideNotification`。源赋值表达式共 3 条仍未纳入此门槛的同名匹配。结论：保留这次语法改进，但拒绝将候选升级为通过；下一步先分别处理实际丢失的事件、名称别名、数组写回及生命周期，再考虑新探针，不运行整批。


## v2b 后续：真实缺失事件处理器的有边界修复

新增通用后处理规则：仅当父组件模板中的 `handleX` 事件处理器未定义、同一个父组件存在源方法 `x`，且 SSM 对应事件恰为 `x` 时，才恢复该绑定；已有声明的 `handleX`、仅在子组件或 computed 块中出现的同名方法、SSM 事件不匹配均不改写。该规则不是对所有 `handle` 前缀进行批量重命名。

对冻结 v2b 候选只读重放后，通知关闭按钮的 `on-click="handleHideNotification"` 改为已有方法 `on-click="hideNotification"`，San 编译通过；源事件回归门槛由 reject 降为 review，新增无法追踪的绑定从 1 降为 0，但 22 条可疑改名和其余静态错误仍在，候选依然不能通过。现有数组克隆规则要求候选本身有明确 `this.data.set('tasks', ...)` 回写；该候选没有，故不机械发明回写。生命周期时序也不自动挪动。完整单测 184/184 通过，未运行模型或浏览器，未修改冻结候选文件。详情见 `data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_v2b_event_alias_offline_replay.json`。
